# FLUX-7B Configurable Training

**Fully configurable FLUX system — adjust any parameter**

## Architecture
```
┌─────────────────────────────────────────────────────────┐
│  RESONANCE FIELD (knowledge storage, no gradients)     │
│  • Size: [H × W × D] × features                        │
│  • inject() → settle() → query()                       │
├─────────────────────────────────────────────────────────┤
│  PHYSICS STACK (frozen)                                │
│  • CSE, wave↔field bridges                            │
├─────────────────────────────────────────────────────────┤
│  EMISSION HEAD (trainable)                             │
│  • Transformer decoder                                 │
│  • Learns HOW TO SPELL from field knowledge           │
└─────────────────────────────────────────────────────────┘
```

## Quick Configs
| Config | Field | Emission | Total | VRAM | Use Case |
|--------|-------|----------|-------|------|----------|
| Tiny | 32³ | 6L/256d | ~50M | 1GB | Testing |
| Small | 64³ | 6L/512d | ~500M | 4GB | Development |
| Medium | 128³ | 12L/768d | ~2B | 16GB | Single GPU |
| Large | 256³ | 12L/1024d | ~4B | 40GB | A100 |
| Full | 512³ | 12L/1024d | ~7B | 80GB | A100 80GB |

---

## Cell 1: Environment Setup

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# ENVIRONMENT DETECTION
# ═══════════════════════════════════════════════════════════════════

import os
from pathlib import Path

IS_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
IS_KAGGLE = os.path.exists('/kaggle')

if IS_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        STORAGE = Path('/content/drive/MyDrive/FLUX_7B')
    except:
        STORAGE = Path('/content/flux_7b')
elif IS_KAGGLE:
    STORAGE = Path('/kaggle/working/flux_7b')
else:
    STORAGE = Path('./flux_7b_output')

STORAGE.mkdir(parents=True, exist_ok=True)
print(f"Environment: {'Colab' if IS_COLAB else 'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Storage: {STORAGE}")

## Cell 2: Clone Repository

In [ ]:
%%time
import os
from pathlib import Path

REPO_URL = 'https://github.com/Unseengap/FLUX.git'

if IS_KAGGLE:
    ROOT = Path('/kaggle/working/FLUX')
elif IS_COLAB:
    ROOT = Path('/content/FLUX')
else:
    ROOT = Path('/Users/admin/Desktop/flux')  # Local path

if not ROOT.exists():
    print('Cloning repo...')
    !git clone {REPO_URL} {ROOT}
else:
    print('Repo exists, pulling latest...')
    !cd {ROOT} && git pull

os.chdir(ROOT)
print(f"Working dir: {os.getcwd()}")

# Install dependencies
!pip install -q -e . 2>/dev/null || pip install -q -r requirements.txt 2>/dev/null
!pip install -q datasets faiss-cpu
print("✓ Dependencies ready")

## Cell 3: Hardware Detection

In [ ]:
import torch
import gc

def get_device():
    if torch.cuda.is_available():
        return 'cuda'
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'

DEVICE = get_device()
print(f"Device: {DEVICE}")

if DEVICE == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
    
    # Auto-suggest config based on VRAM
    if vram_gb >= 70:
        SUGGESTED_CONFIG = 'full'
    elif vram_gb >= 35:
        SUGGESTED_CONFIG = 'large'
    elif vram_gb >= 14:
        SUGGESTED_CONFIG = 'medium'
    elif vram_gb >= 6:
        SUGGESTED_CONFIG = 'small'
    else:
        SUGGESTED_CONFIG = 'tiny'
    print(f"\nSuggested config: {SUGGESTED_CONFIG}")
else:
    SUGGESTED_CONFIG = 'tiny'
    print(f"\nNo GPU detected. Using config: {SUGGESTED_CONFIG}")

---
# ⚙️ CONFIGURATION — EDIT THIS CELL
---

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                    ⚙️ MASTER CONFIGURATION ⚙️
# ═══════════════════════════════════════════════════════════════════
#
# EDIT THESE VALUES TO CUSTOMIZE YOUR FLUX-7B SYSTEM
#
# ═══════════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────────────
# PRESET SELECTION (or set to None and customize below)
# ─────────────────────────────────────────────────────────────────────
# Options: 'tiny', 'small', 'medium', 'large', 'full', or None
USE_PRESET = 'tiny'  # ← Change this or set to None for custom

# ─────────────────────────────────────────────────────────────────────
# FIELD CONFIGURATION (Knowledge Storage)
# ─────────────────────────────────────────────────────────────────────
FIELD_SIZE = 32          # Field dimensions: [SIZE × SIZE × SIZE]
                         # 32=tiny, 64=small, 128=medium, 256=large, 512=full
                         
FIELD_FEATURES = 512     # Features per location
                         # 512=tiny/small, 768=medium, 1024=large, 1536=full

# ─────────────────────────────────────────────────────────────────────
# EMISSION HEAD CONFIGURATION (Trainable Part)
# ─────────────────────────────────────────────────────────────────────
EMISSION_D_MODEL = 256   # Model dimension
                         # 256=tiny, 512=small, 768=medium, 1024=large/full
                         
EMISSION_LAYERS = 4      # Number of transformer layers
                         # 4=tiny, 6=small, 12=medium/large/full
                         
EMISSION_HEADS = 4       # Number of attention heads
                         # 4=tiny, 8=small, 12=medium, 16=large/full

EMISSION_FFN_DIM = 1024  # FFN intermediate dimension (usually 4× d_model)
                         # 1024=tiny, 2048=small, 3072=medium, 4096=large/full

# ─────────────────────────────────────────────────────────────────────
# PHYSICS CONFIGURATION
# ─────────────────────────────────────────────────────────────────────
WAVE_DIM = 432           # CSE wave dimension (keep at 432 for compatibility)
K_ATTRACTORS = 16        # Number of field attractors to retrieve per query

# ─────────────────────────────────────────────────────────────────────
# INJECTION CONFIGURATION
# ─────────────────────────────────────────────────────────────────────
INJECTION_STRENGTH = 1.0   # How strongly to inject knowledge (0.5-2.0)
INJECTION_RADIUS = 4       # Neighborhood radius for injection
SETTLE_STEPS = 50          # Energy minimization steps after injection
SETTLE_EVERY = 100         # Settle after every N documents

# ─────────────────────────────────────────────────────────────────────
# TRAINING CONFIGURATION
# ─────────────────────────────────────────────────────────────────────
LEARNING_RATE = 1e-4       # Initial learning rate
MIN_LEARNING_RATE = 1e-6   # Final learning rate (cosine decay)
WARMUP_STEPS = 500         # LR warmup steps
TOTAL_STEPS = 10000        # Total training steps
BATCH_SIZE = 4             # Batch size per step
GRAD_ACCUM = 4             # Gradient accumulation steps
MAX_SEQ_LEN = 256          # Maximum sequence length
DROPOUT = 0.1              # Dropout rate
LABEL_SMOOTHING = 0.1      # Label smoothing

# ─────────────────────────────────────────────────────────────────────
# DATA CONFIGURATION
# ─────────────────────────────────────────────────────────────────────
INJECTION_DOCS = 1000      # Number of documents to inject
MIN_DOC_LENGTH = 50        # Minimum document length (bytes)
MAX_DOC_LENGTH = 2048      # Maximum document length (bytes)

# ─────────────────────────────────────────────────────────────────────
# LOGGING & CHECKPOINTS
# ─────────────────────────────────────────────────────────────────────
LOG_EVERY = 100            # Log every N steps
EVAL_EVERY = 500           # Evaluate every N steps
CHECKPOINT_EVERY = 2000    # Save checkpoint every N steps

print("✓ Configuration loaded")
print(f"  Preset: {USE_PRESET if USE_PRESET else 'Custom'}")

## Cell 5: Apply Preset (if selected)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# PRESET CONFIGURATIONS
# ═══════════════════════════════════════════════════════════════════

PRESETS = {
    'tiny': {
        'FIELD_SIZE': 32,
        'FIELD_FEATURES': 512,
        'EMISSION_D_MODEL': 256,
        'EMISSION_LAYERS': 4,
        'EMISSION_HEADS': 4,
        'EMISSION_FFN_DIM': 1024,
        'K_ATTRACTORS': 8,
        'TOTAL_STEPS': 1000,
        'INJECTION_DOCS': 500,
    },
    'small': {
        'FIELD_SIZE': 64,
        'FIELD_FEATURES': 512,
        'EMISSION_D_MODEL': 512,
        'EMISSION_LAYERS': 6,
        'EMISSION_HEADS': 8,
        'EMISSION_FFN_DIM': 2048,
        'K_ATTRACTORS': 16,
        'TOTAL_STEPS': 10000,
        'INJECTION_DOCS': 5000,
    },
    'medium': {
        'FIELD_SIZE': 128,
        'FIELD_FEATURES': 768,
        'EMISSION_D_MODEL': 768,
        'EMISSION_LAYERS': 12,
        'EMISSION_HEADS': 12,
        'EMISSION_FFN_DIM': 3072,
        'K_ATTRACTORS': 24,
        'TOTAL_STEPS': 50000,
        'INJECTION_DOCS': 50000,
    },
    'large': {
        'FIELD_SIZE': 256,
        'FIELD_FEATURES': 1024,
        'EMISSION_D_MODEL': 1024,
        'EMISSION_LAYERS': 12,
        'EMISSION_HEADS': 16,
        'EMISSION_FFN_DIM': 4096,
        'K_ATTRACTORS': 32,
        'TOTAL_STEPS': 100000,
        'INJECTION_DOCS': 500000,
    },
    'full': {
        'FIELD_SIZE': 512,
        'FIELD_FEATURES': 1536,
        'EMISSION_D_MODEL': 1024,
        'EMISSION_LAYERS': 12,
        'EMISSION_HEADS': 16,
        'EMISSION_FFN_DIM': 4096,
        'K_ATTRACTORS': 32,
        'TOTAL_STEPS': 100000,
        'INJECTION_DOCS': 1000000,
    },
}

# Apply preset if selected
if USE_PRESET and USE_PRESET in PRESETS:
    preset = PRESETS[USE_PRESET]
    for key, value in preset.items():
        globals()[key] = value
    print(f"✓ Applied preset: {USE_PRESET}")
else:
    print("✓ Using custom configuration")

# Calculate and display stats
field_params = FIELD_SIZE ** 3 * FIELD_FEATURES
emission_params = EMISSION_LAYERS * (4 * EMISSION_D_MODEL ** 2 + 2 * EMISSION_D_MODEL * EMISSION_FFN_DIM)
total_params = field_params + emission_params

print(f"\n═══ CONFIGURATION SUMMARY ═══")
print(f"Field: [{FIELD_SIZE}³] × {FIELD_FEATURES} = {field_params/1e6:.0f}M params")
print(f"Emission: {EMISSION_LAYERS}L × {EMISSION_D_MODEL}d × {EMISSION_HEADS}h = {emission_params/1e6:.0f}M params")
print(f"Total: {total_params/1e9:.2f}B params")
print(f"Training steps: {TOTAL_STEPS:,}")
print(f"Injection docs: {INJECTION_DOCS:,}")

# Estimate memory
mem_gb = (field_params * 4 + emission_params * 4) / 1e9
print(f"Estimated VRAM: {mem_gb:.1f} GB")

## Cell 6: Build FLUX System

In [ ]:
import sys
from pathlib import Path

# Add paths
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'phases' / 'flux_7b'))
for phase in ['phase1', 'phase2', 'phase3', 'phase8']:
    sys.path.insert(0, str(ROOT / 'phases' / phase))

print("Building FLUX system with custom configuration...")

from dataclasses import dataclass
import torch
import torch.nn as nn

# ─────────────────────────────────────────────────────────────────────
# CUSTOM FIELD CONFIG
# ─────────────────────────────────────────────────────────────────────
@dataclass
class CustomFieldConfig:
    size_x: int = FIELD_SIZE
    size_y: int = FIELD_SIZE
    size_z: int = FIELD_SIZE
    features: int = FIELD_FEATURES
    injection_radius: int = INJECTION_RADIUS
    settle_rate: float = 0.01
    mass_growth: float = 0.1
    cooling_rate: float = 0.99
    attractor_threshold: float = 0.5
    index_nlist: int = 1024
    
    @property
    def total_params(self):
        return self.size_x * self.size_y * self.size_z * self.features

# ─────────────────────────────────────────────────────────────────────
# CUSTOM EMISSION CONFIG
# ─────────────────────────────────────────────────────────────────────
@dataclass 
class CustomEmissionConfig:
    field_features: int = FIELD_FEATURES
    wave_dim: int = WAVE_DIM
    d_model: int = EMISSION_D_MODEL
    num_layers: int = EMISSION_LAYERS
    num_heads: int = EMISSION_HEADS
    d_ff: int = EMISSION_FFN_DIM
    vocab_size: int = 256
    max_seq_len: int = MAX_SEQ_LEN
    dropout: float = DROPOUT
    k_attractors: int = K_ATTRACTORS

print(f"  Field config: {FIELD_SIZE}³ × {FIELD_FEATURES}")
print(f"  Emission config: {EMISSION_LAYERS}L × {EMISSION_D_MODEL}d × {EMISSION_HEADS}h")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# LOAD FLUX-APEX MODEL (CSE + Bridges)
# ─────────────────────────────────────────────────────────────────────

from huggingface_hub import hf_hub_download

# Download Flux-Apex-V1 from HuggingFace (contains trained CSE)
print("═══ LOADING FLUX-APEX-V1 ═══")
print("Downloading from HuggingFace (contains trained CSE + bridges)...")

apex_path = ROOT / 'checkpoints' / 'Flux-Apex-V1.flx'
if not apex_path.exists():
    apex_path = Path(hf_hub_download(
        repo_id='UnseenGAP/FLUX',
        filename='checkpoints/Flux-Apex-V1.flx',
        local_dir=ROOT,
    ))
    print(f"  ✓ Downloaded to {apex_path}")
else:
    print(f"  ✓ Found local: {apex_path}")

# Load the .flx archive
print("\nLoading Flux-Apex-V1.flx archive...")
apex_state = torch.load(str(apex_path), map_location='cpu', weights_only=False)

print(f"  Format: {apex_state.get('format', 'unknown')}")
print(f"  Version: {apex_state.get('version', 'unknown')}")
print(f"  Phase: {apex_state.get('phase', 'unknown')}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# EXTRACT CSE FROM FLUX-APEX
# ─────────────────────────────────────────────────────────────────────

sys.path.insert(0, str(ROOT / 'phases' / 'phase1'))
from cse import ContinuousSemanticEncoder

print("Loading CSE (Continuous Semantic Encoder) from Flux-Apex...")

# Get CSE config and state_dict from the archive
cse_section = apex_state['cse']
cse_config = cse_section.get('config', {})
cse_state_dict = cse_section['state_dict']

print(f"  CSE config: {cse_config}")
print(f"  CSE tensors: {len(cse_state_dict)}")

# Create CSE with matching config
cse = ContinuousSemanticEncoder(
    wave_dims=cse_config.get('wave_dims', {
        'phonetic': 64,
        'syntactic': 64, 
        'semantic': 256,
        'temporal': 32,
        'intensity': 16,
    }),
    byte_window=cse_config.get('byte_window', 8),
    byte_stride=cse_config.get('byte_stride', 1),
    interference_radius=cse_config.get('interference_radius', 4),
    device='cpu',
)

# Load trained weights
cse.load_state_dict(cse_state_dict)
cse = cse.to(DEVICE)
cse.eval()

# Freeze CSE (it's already trained)
for p in cse.parameters():
    p.requires_grad = False

cse_params = sum(p.numel() for p in cse.parameters())
print(f"  ✓ CSE loaded: {cse_params:,} params (frozen)")

# Test encoding
test_wave = cse.encode("Hello, FLUX!")
print(f"  ✓ Test encoding: {test_wave.full.shape}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# EXTRACT BRIDGES FROM FLUX-APEX
# ─────────────────────────────────────────────────────────────────────

print("Loading bridges (wave↔field projections) from Flux-Apex...")

bridges_section = apex_state.get('bridges', {})
bridges_state = bridges_section.get('state_dict', bridges_section)

# Create physics bridge (wave_to_field + field_to_wave)
class PhysicsBridge(nn.Module):
    """Project waves to field space and back. Uses trained weights from Flux-Apex."""
    
    def __init__(self, wave_dim=432, field_features=512):
        super().__init__()
        self.wave_to_field = nn.Linear(wave_dim, field_features)
        self.field_to_wave = nn.Linear(field_features, wave_dim)
        
physics = PhysicsBridge(WAVE_DIM, FIELD_FEATURES).to(DEVICE)

# Try to load trained bridge weights from Flux-Apex
if 'wave_to_field' in bridges_state:
    wtf_state = bridges_state['wave_to_field']
    if isinstance(wtf_state, dict) and 'weight' in wtf_state:
        # Adjust if dimensions differ
        if wtf_state['weight'].shape == physics.wave_to_field.weight.shape:
            physics.wave_to_field.load_state_dict(wtf_state)
            print(f"  ✓ wave_to_field loaded: {wtf_state['weight'].shape}")
        else:
            print(f"  ⚠ wave_to_field shape mismatch: {wtf_state['weight'].shape} vs {physics.wave_to_field.weight.shape}")
            print(f"    Using random initialization (will project from CSE output)")

if 'field_to_wave' in bridges_state:
    ftw_state = bridges_state['field_to_wave']
    if isinstance(ftw_state, dict) and 'weight' in ftw_state:
        if ftw_state['weight'].shape == physics.field_to_wave.weight.shape:
            physics.field_to_wave.load_state_dict(ftw_state)
            print(f"  ✓ field_to_wave loaded: {ftw_state['weight'].shape}")
        else:
            print(f"  ⚠ field_to_wave shape mismatch, using projection")

physics.eval()
for p in physics.parameters():
    p.requires_grad = False
    
print(f"  ✓ Physics bridge ready")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# BUILD FIELD AND EMISSION HEAD
# ─────────────────────────────────────────────────────────────────────

from resonance_field_7b import ResonanceField7B
from emission_head import EmissionHead

# Create field with custom config
field_config = CustomFieldConfig()
print(f"\nCreating field: {field_config.total_params/1e6:.0f}M params...")
field = ResonanceField7B(field_config, device=DEVICE)
print(f"  ✓ Field created")

# Create emission head with custom config
emission_config = CustomEmissionConfig()
print(f"\nCreating emission head...")
emission = EmissionHead(emission_config).to(DEVICE)
emission_params = sum(p.numel() for p in emission.parameters())
print(f"  ✓ Emission head: {emission_params/1e6:.1f}M params")

# Total
total = field_config.total_params + emission_params + cse_params
print(f"\n═══ SYSTEM READY ═══")
print(f"Field: {field_config.total_params/1e6:.0f}M params (no gradients)")
print(f"CSE: {cse_params/1e6:.1f}M params (frozen, from Flux-Apex)")
print(f"Emission: {emission_params/1e6:.1f}M params (trainable)")
print(f"Total: {total/1e9:.2f}B")

## Cell 8: Load Data

In [ ]:
from datasets import load_dataset

print(f"Loading {INJECTION_DOCS:,} documents for injection...")

# Load OpenWebText subset
ds = load_dataset("openwebtext", split=f"train[:{INJECTION_DOCS}]")

# Filter by length
documents = [
    doc['text'][:MAX_DOC_LENGTH] 
    for doc in ds 
    if MIN_DOC_LENGTH < len(doc['text'])
]

print(f"  ✓ Loaded {len(documents):,} documents")
print(f"  Sample: {documents[0][:100]}...")

## Cell 9: Knowledge Injection (No Gradients)

In [ ]:
from tqdm import tqdm
import time

print("═══ KNOWLEDGE INJECTION ═══")
print("Using TRAINED CSE from Flux-Apex for encoding")
print("This uses NO gradients — direct field manipulation")
print()

start_time = time.time()

for i, doc in enumerate(tqdm(documents, desc="Injecting")):
    try:
        # === USE REAL CSE FROM FLUX-APEX ===
        # Encode document using trained CSE (outputs SemanticWave)
        with torch.no_grad():
            semantic_wave = cse.encode(doc[:MAX_DOC_LENGTH])
            
            # semantic_wave.full has shape [seq_len, 432]
            # Take mean across sequence for document-level representation
            doc_wave = semantic_wave.full.mean(dim=0)  # [432]
            
            # Project to field space using trained bridge
            field_wave = physics.wave_to_field(doc_wave)  # [field_features]
        
        # Inject into field (no gradients)
        field.inject(field_wave, strength=INJECTION_STRENGTH)
        
        # Periodic settling
        if (i + 1) % SETTLE_EVERY == 0:
            field.settle(SETTLE_STEPS)
            
    except Exception as e:
        if i < 5:  # Only print first few errors
            print(f"  ⚠ Error on doc {i}: {e}")
        continue

# Final settling
print("\nFinal settling...")
field.settle(SETTLE_STEPS * 2)

elapsed = time.time() - start_time
print(f"\n✓ Injection complete in {elapsed:.1f}s")
print(f"  Documents: {len(documents):,}")
print(f"  Rate: {len(documents)/elapsed:.1f} docs/sec")
print(f"  CSE encoding: ✓ (from Flux-Apex-V1)")
print(f"  Field stats: {field.stats()}")

## Cell 10: Train Emission Head

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.nn.functional as F

print("═══ EMISSION TRAINING ═══")
print(f"Training {emission_params/1e6:.1f}M parameters")
print(f"Steps: {TOTAL_STEPS:,}")
print()

# Optimizer (emission only)
optimizer = AdamW(
    emission.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01,
    betas=(0.9, 0.95),
)

scheduler = CosineAnnealingLR(
    optimizer,
    T_max=TOTAL_STEPS - WARMUP_STEPS,
    eta_min=MIN_LEARNING_RATE,
)

# Training state
emission.train()
global_step = 0
total_loss = 0.0
best_loss = float('inf')
start_time = time.time()

print(f"Starting training...\n")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# TRAINING LOOP (Using trained CSE from Flux-Apex)
# ─────────────────────────────────────────────────────────────────────

import random

optimizer.zero_grad()
accum_loss = 0.0

pbar = tqdm(total=TOTAL_STEPS, desc="Training")

while global_step < TOTAL_STEPS:
    # Sample random document
    doc = random.choice(documents)
    doc_text = doc[:MAX_SEQ_LEN]
    doc_bytes = list(doc_text.encode('utf-8'))
    
    if len(doc_bytes) < 10:
        continue
        
    try:
        # Create target bytes
        target = torch.tensor(doc_bytes, dtype=torch.long, device=DEVICE)
        
        # === USE REAL CSE FROM FLUX-APEX ===
        with torch.no_grad():
            semantic_wave = cse.encode(doc_text)
            # wave_context: [1, seq_len, 432] from CSE
            wave_context = semantic_wave.full.unsqueeze(0)  # [1, seq, 432]
        
        # Query field for context
        with torch.no_grad():
            query_wave = physics.wave_to_field(wave_context.mean(dim=1).squeeze(0))
            field_features, gravity_weights, _ = field.query(query_wave, k=K_ATTRACTORS)
        
        # Forward pass
        logits = emission(
            field_features.unsqueeze(0),
            gravity_weights.unsqueeze(0),
            wave_context,
            target.unsqueeze(0),
        )  # [1, seq, 256]
        
        # Loss
        loss = F.cross_entropy(
            logits.view(-1, 256),
            target.view(-1),
            label_smoothing=LABEL_SMOOTHING,
        )
        
        # Backward
        loss = loss / GRAD_ACCUM
        loss.backward()
        accum_loss += loss.item() * GRAD_ACCUM
        
        # Step
        if (global_step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(emission.parameters(), 1.0)
            
            # LR warmup
            if global_step < WARMUP_STEPS:
                lr = LEARNING_RATE * (global_step + 1) / WARMUP_STEPS
                for pg in optimizer.param_groups:
                    pg['lr'] = lr
            else:
                scheduler.step()
                
            optimizer.step()
            optimizer.zero_grad()
            
            total_loss += accum_loss / GRAD_ACCUM
            
            # Logging
            if (global_step + 1) % LOG_EVERY == 0:
                avg_loss = total_loss / (global_step + 1)
                lr = optimizer.param_groups[0]['lr']
                pbar.set_postfix({
                    'loss': f'{accum_loss/GRAD_ACCUM:.4f}',
                    'avg': f'{avg_loss:.4f}',
                    'lr': f'{lr:.2e}',
                })
                
            # Checkpoint
            if (global_step + 1) % CHECKPOINT_EVERY == 0:
                checkpoint_path = STORAGE / f'emission_step{global_step+1}.pt'
                torch.save({
                    'step': global_step + 1,
                    'emission_state_dict': emission.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': accum_loss / GRAD_ACCUM,
                }, checkpoint_path)
                print(f"\n  ✓ Checkpoint: {checkpoint_path}")
                
            accum_loss = 0.0
            
        global_step += 1
        pbar.update(1)
        
    except Exception as e:
        continue

pbar.close()

elapsed = time.time() - start_time
print(f"\n═══ TRAINING COMPLETE ═══")
print(f"Steps: {global_step:,}")
print(f"Final loss: {total_loss / max(1, global_step):.4f}")
print(f"Time: {elapsed/60:.1f} minutes")

## Cell 12: Test Generation

In [ ]:
print("═══ GENERATION TEST ═══\n")

emission.eval()

test_prompts = [
    "The capital of France is",
    "Machine learning is",
    "The best way to learn programming is",
    "In the future, artificial intelligence will",
]

for prompt in test_prompts:
    # Create wave context from prompt
    prompt_bytes = list(prompt.encode('utf-8'))
    wave_context = torch.randn(1, len(prompt_bytes), WAVE_DIM, device=DEVICE) * 0.1
    
    # Query field
    with torch.no_grad():
        query_wave = physics.wave_to_field(wave_context.mean(dim=1).squeeze(0))
        field_features, gravity_weights, _ = field.query(query_wave, k=K_ATTRACTORS)
        
        # Generate
        output = emission.generate(
            field_features.unsqueeze(0),
            gravity_weights.unsqueeze(0),
            wave_context,
            max_length=100,
            temperature=0.8,
        )
        
    generated = output.decode('utf-8', errors='replace')
    print(f"Prompt: {prompt}")
    print(f"Output: {generated}")
    print("-" * 60)

## Cell 13: Save Final System

In [ ]:
print("Saving final system...")

# Save field
field.save(str(STORAGE / 'field_final.pt'))

# Save emission
torch.save({
    'emission_state_dict': emission.state_dict(),
    'config': {
        'field_size': FIELD_SIZE,
        'field_features': FIELD_FEATURES,
        'emission_d_model': EMISSION_D_MODEL,
        'emission_layers': EMISSION_LAYERS,
        'emission_heads': EMISSION_HEADS,
        'total_steps': global_step,
    },
}, STORAGE / 'emission_final.pt')

print(f"\n✓ System saved to {STORAGE}")
print(f"  - field_final.pt")
print(f"  - emission_final.pt")

## Cell 14: Interactive Playground

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# INTERACTIVE PLAYGROUND
# ═══════════════════════════════════════════════════════════════════
# Change the prompt and run this cell to generate!

PROMPT = "The meaning of life is"  # ← EDIT THIS
MAX_LENGTH = 150
TEMPERATURE = 0.8

# Generate
emission.eval()
prompt_bytes = list(PROMPT.encode('utf-8'))
wave_context = torch.randn(1, len(prompt_bytes), WAVE_DIM, device=DEVICE) * 0.1

with torch.no_grad():
    query_wave = physics.wave_to_field(wave_context.mean(dim=1).squeeze(0))
    field_features, gravity_weights, _ = field.query(query_wave, k=K_ATTRACTORS)
    
    output = emission.generate(
        field_features.unsqueeze(0),
        gravity_weights.unsqueeze(0),
        wave_context,
        max_length=MAX_LENGTH,
        temperature=TEMPERATURE,
    )

print(f"Prompt: {PROMPT}")
print(f"\nGenerated:")
print(output.decode('utf-8', errors='replace'))